# 02. Vital Sign Raw (활력징후 원본 추출)

## 목적
코호트 환자의 활력징후 데이터를 **1시간 단위**로 추출 (슬라이딩 윈도우 적용 전)

## 입력
- `cohort_base.csv`: 기본 코호트 (환자 목록)

## 출력
- `vital_raw.csv`: 시간별 활력징후 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (chartevents)
- [x] 타입 변환 (온도 F→C)
- [x] 이상치 클리핑 (임상 범위)
- [ ] 결측치 처리 → preprocessing 단계에서
- [ ] 통계량 계산 → feature engineering 단계에서

## Item IDs
| 항목 | Item ID | 비고 |
|------|---------|------|
| HR | 220045 | Heart Rate |
| RR | 220210, 224690 | Respiratory Rate |
| SpO2 | 220277 | Oxygen Saturation |
| Temp | 223761 (F), 223762 (C) | Temperature |
| NBP | 220179, 220180, 220181 | Non-invasive BP (Sys/Dia/Mean) |
| ABP | 220050, 220051, 220052 | Arterial BP (Sys/Dia/Mean) |

In [ ]:
import duckdb
import pandas as pd
import os

# 설정
DB_PATH = '../data/duckdb/mimic_total.duckdb'
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

# DuckDB 연결
con = duckdb.connect(DB_PATH)
print("=== 02. Vital Sign Raw 추출 시작 ===")

## Step 1: 코호트 로드

In [ ]:
print("Step 1: 코호트 로드")

cohort_path = os.path.join(INPUT_DIR, 'cohort_base.csv')
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

# DuckDB에 등록
con.register('cohort', df_cohort)

## Step 2: Vital Sign 추출 (1시간 단위 집계)

In [ ]:
print("\nStep 2: Vital Sign 추출 (1시간 단위)")

# 임상 범위 정의 (이상치 클리핑용)
CLINICAL_RANGES = {
    'hr': (20, 300),
    'rr': (4, 60),
    'spo2': (50, 100),
    'temp': (30, 45),      # Celsius
    'sbp': (40, 300),
    'dbp': (20, 200),
    'mbp': (30, 250),
}

vital_query = """
WITH base_vitals AS (
    SELECT
        c.stay_id,
        date_trunc('hour', CAST(ce.charttime AS TIMESTAMP)) as charttime_h,
        ce.itemid,
        -- 온도 F→C 변환
        CASE 
            WHEN ce.itemid = '223761' THEN (TRY_CAST(ce.valuenum AS DOUBLE) - 32) * 5.0 / 9.0
            ELSE TRY_CAST(ce.valuenum AS DOUBLE)
        END as val_num,
        CAST(ce.charttime AS TIMESTAMP) as charttime_ts
    FROM cohort c
    INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
    WHERE ce.itemid IN (
        '220045',                       -- HR
        '220210', '224690',             -- RR
        '220277',                       -- SpO2
        '223761', '223762',             -- Temp (F, C)
        '220179', '220180', '220181',   -- NBP (Sys, Dia, Mean)
        '220050', '220051', '220052'    -- ABP (Sys, Dia, Mean)
    )
    AND TRY_CAST(ce.valuenum AS DOUBLE) IS NOT NULL
    AND TRY_CAST(ce.valuenum AS DOUBLE) > 0
    -- ICU 체류 기간 내 데이터만
    AND CAST(ce.charttime AS TIMESTAMP) >= c.intime
    AND CAST(ce.charttime AS TIMESTAMP) <= c.outtime
),
aggregated AS (
    SELECT
        stay_id,
        charttime_h,
        
        -- HR: 중간값
        MEDIAN(CASE WHEN itemid = '220045' THEN val_num END) as hr,
        
        -- RR: 중간값
        MEDIAN(CASE WHEN itemid IN ('220210', '224690') THEN val_num END) as rr,
        
        -- SpO2: 최저값 (위험 신호)
        MIN(CASE WHEN itemid = '220277' THEN val_num END) as spo2,
        
        -- Temp: 최신값
        MAX_BY(val_num, charttime_ts) FILTER (WHERE itemid IN ('223761', '223762')) as temp,
        
        -- BP: NBP 우선, 없으면 ABP (최저값 - 위험 신호)
        COALESCE(
            MIN(CASE WHEN itemid = '220179' THEN val_num END),
            MIN(CASE WHEN itemid = '220050' THEN val_num END)
        ) as sbp,
        COALESCE(
            MIN(CASE WHEN itemid = '220180' THEN val_num END),
            MIN(CASE WHEN itemid = '220051' THEN val_num END)
        ) as dbp,
        COALESCE(
            MIN(CASE WHEN itemid = '220181' THEN val_num END),
            MIN(CASE WHEN itemid = '220052' THEN val_num END)
        ) as mbp
        
    FROM base_vitals
    GROUP BY stay_id, charttime_h
)
SELECT
    stay_id,
    charttime_h,
    
    -- 이상치 클리핑 적용
    CASE WHEN hr BETWEEN 20 AND 300 THEN hr ELSE NULL END as hr,
    CASE WHEN rr BETWEEN 4 AND 60 THEN rr ELSE NULL END as rr,
    CASE WHEN spo2 BETWEEN 50 AND 100 THEN spo2 ELSE NULL END as spo2,
    CASE WHEN temp BETWEEN 30 AND 45 THEN temp ELSE NULL END as temp,
    CASE WHEN sbp BETWEEN 40 AND 300 THEN sbp ELSE NULL END as sbp,
    CASE WHEN dbp BETWEEN 20 AND 200 THEN dbp ELSE NULL END as dbp,
    CASE WHEN mbp BETWEEN 30 AND 250 THEN mbp ELSE NULL END as mbp
    
FROM aggregated
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중... (수 분 소요될 수 있음)")
df_vital = con.execute(vital_query).df()

print(f"✓ Vital Sign 추출 완료")
print(f"  - 총 행 수: {len(df_vital):,}개")
print(f"  - 고유 환자: {df_vital['stay_id'].nunique():,}명")

## Step 3: 데이터 품질 확인

In [ ]:
print("\nStep 3: 데이터 품질 확인")

# 결측치 비율
print("\n=== 결측치 비율 ===")
vital_cols = ['hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp']
for col in vital_cols:
    missing_rate = df_vital[col].isna().mean() * 100
    print(f"  {col.upper()}: {missing_rate:.1f}% 결측")

# 기술 통계
print("\n=== 기술 통계 ===")
df_vital[vital_cols].describe().round(2)

## Step 4: 저장

In [ ]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = os.path.join(OUTPUT_DIR, 'vital_raw.csv')
df_vital.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: vital_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_vital):,}개 (1 row = 1 patient × 1 hour)")
print(f"  - 경로: {output_path}")

In [ ]:
# 샘플 확인
print("\n=== 샘플 데이터 ===")
df_vital.head(10)

In [ ]:
con.close()
print("\n=== 02. Vital Sign Raw 추출 완료 ===")